# Bonus — Generate Rich Document Images

**Hands-on synthetic data pipeline workshop with NeMo Data Designer and NeMo Anonymizer — PyData London 2026**

This bonus notebook uses Data Designer's image generation capabilities to create
synthetic business document images with rich visual information: readable text,
tables, charts, KPI cards, timelines, callouts, signatures, and scan artifacts.
It uses Gemini image generation via OpenRouter to produce varied document pages
with controlled metadata.

## What it produces

A parquet file at `data/rich_document_seed.parquet` containing base64-encoded
document images plus four orientation fields: `primary_visual`,
`secondary_visual`, `layout_style`, and `document_condition`. This is an
optional exploratory seed dataset; the core workshop notebooks do not depend on
running this bonus notebook.

## What you'll learn

- **Image generation columns** -- `ImageColumnConfig` with a prompt template
- **Visual diversity controls** -- sampler columns for document type, chart type, table type, and layout
- **Persona-driven stakeholders** -- Nemotron Personas for realistic synthetic owner/contact fields
- **Rich document composition** -- text, charts, tables, diagrams, callouts, stamps, and annotations
- **Quality factors** -- scan quality, rotation, mobile photos, faded print, and markup
- **Labeled outputs** -- document images paired with known generation metadata

> **Prerequisites**: This optional notebook requires `OPENROUTER_API_KEY` for image generation and `NGC_API_KEY` if you need to download Nemotron Personas.


## Setup

In [ ]:
from notebook_helpers import (
    DATA_DIR,
    environment_setup,
)

provider = environment_setup(provider="openrouter")

### Download persona dataset

The `PersonSampler` generates realistic synthetic stakeholder names and contact fields from the [Nemotron Personas](https://nvidia-nemo.github.io/DataDesigner/latest/concepts/person_sampling/) dataset hosted on NGC (~1.2 GB for `en_US`). The download requires an `NGC_API_KEY` environment variable. Skip the cell below if the dataset is already downloaded.


In [ ]:
# !yes | uv run data-designer download personas --locale en_US

### Configure image generation model

In [ ]:
import data_designer.config as dd
from data_designer.interface import DataDesigner

data_designer = DataDesigner(
    model_providers=[
        dd.ModelProvider(
            name=provider.provider_name,
            endpoint=provider.endpoint,
            provider_type="openai",
            api_key=provider.env_var,
        )
    ]
)

In [ ]:
MODEL_PROVIDER = "openrouter"
MODEL_ID = "google/gemini-3.1-flash-image-preview"
MODEL_ALIAS = "document-generation-model"

model_configs = [
    dd.ModelConfig(
        alias=MODEL_ALIAS,
        model=MODEL_ID,
        provider=MODEL_PROVIDER,
        inference_parameters=dd.ImageInferenceParams(
            extra_body={
                "generationConfig": {
                    "imageConfig": {
                        "aspectRatio": "2:3",
                        "imageSize": "1024"
                    }
                }
            },
            max_parallel_requests=10
        ),
    ),
]

## Build configuration: Rich document generation

We'll generate synthetic business documents with controlled variation across:
- **Synthetic stakeholders**: persona-grounded document owners with names, locations, and contact fields
- **Document families**: reports, dashboards, research briefs, statements, memos, audits, and one-page plans
- **Visual elements**: charts, tables, KPI cards, timelines, maps, diagrams, and callouts
- **Layout styles**: dense analytical layouts, executive summaries, compliance forms, board slides, and annotated printouts
- **Content themes**: revenue, operations, risk, support, capacity, sustainability, and hiring
- **Quality factors**: pristine exports, scans, phone photos, faded print, markup, and light rotation


In [ ]:
config_builder = dd.DataDesignerConfigBuilder(model_configs=model_configs)


def add_category(name, values, weights=None):
    config_builder.add_column(
        dd.SamplerColumnConfig(
            name=name,
            sampler_type=dd.SamplerType.CATEGORY,
            params=dd.CategorySamplerParams(values=values, weights=weights),
        )
    )


add_category(
    "document_type",
    [
        "quarterly business review",
        "market research brief",
        "operations dashboard export",
        "clinical trial status report",
        "sustainability impact report",
        "financial variance memo",
        "customer support incident review",
        "supply chain risk assessment",
        "product launch readiness plan",
        "employee engagement summary",
    ],
    weights=[0.12, 0.10, 0.14, 0.08, 0.08, 0.12, 0.12, 0.10, 0.12, 0.12],
)

add_category(
    "organization_name",
    [
        "Aster Analytics",
        "Blue Ridge Health",
        "CedarWorks Manufacturing",
        "DeltaGrid Energy",
        "Evergreen Mobility",
        "Harborlight Retail",
        "Northstar Robotics",
        "Redwood BioSystems",
        "Summit Cloud Services",
        "Valley Forge Logistics",
    ],
)

add_category(
    "audience",
    [
        "executive leadership",
        "finance review committee",
        "field operations managers",
        "clinical program leads",
        "board audit committee",
        "customer success directors",
    ],
)

config_builder.add_column(
    dd.SamplerColumnConfig(
        name="stakeholder",
        sampler_type=dd.SamplerType.PERSON,
        params=dd.PersonSamplerParams(
            locale="en_US",
            age_range=[24, 72],
            with_synthetic_personas=True,
        ),
    )
)

add_category(
    "stakeholder_role",
    [
        "VP Operations",
        "Finance Director",
        "Clinical Program Manager",
        "Customer Success Lead",
        "Risk Officer",
        "Product Launch Owner",
        "People Analytics Partner",
    ],
)

add_category(
    "content_theme",
    [
        "quarterly revenue performance and forecast variance",
        "regional customer adoption and churn risk",
        "service-level agreement compliance and incident aging",
        "inventory throughput, backorders, and supplier delays",
        "trial enrollment, site activation, and adverse event counts",
        "energy consumption, emissions, and sustainability targets",
        "hiring funnel conversion, offer acceptance, and attrition",
        "product launch milestones, owners, and readiness status",
    ],
)

add_category(
    "primary_visual",
    [
        "clustered bar chart comparing three regions across four quarters",
        "line chart with two series, annotated inflection points, and a target band",
        "stacked area chart showing category mix over six months",
        "waterfall chart showing contributors to budget variance",
        "scatter plot with labeled outliers and a trend line",
        "Gantt-style timeline with milestones and owner initials",
        "heatmap matrix with risk severity by team and region",
        "donut chart with callout labels and percentages",
    ],
)

add_category(
    "secondary_visual",
    [
        "dense financial table with subtotals and variance arrows",
        "KPI card row with current value, target, delta, and traffic-light status",
        "two-column risk register with owner, due date, and mitigation note",
        "small process diagram with arrows between four labeled stages",
        "ranked list table with sparklines in the final column",
        "compact map inset with region labels and numeric badges",
        "executive callout box with three bullet conclusions",
        "signature block plus approval checklist",
    ],
)

add_category(
    "layout_style",
    [
        "clean consulting report page with narrow margins and section dividers",
        "dashboard export with a top filter bar and grid of panels",
        "formal memo with letterhead, dense paragraphs, and one embedded chart",
        "board-pack page with title ribbon, footnotes, and small-print source notes",
        "compliance form with checkboxes, tables, and stamped approval",
        "research brief with abstract, sidebar definitions, and figure captions",
        "operations one-pager with color-coded status chips and action table",
    ],
)

add_category(
    "document_condition",
    [
        "pristine exported PDF screenshot",
        "high-resolution office scanner output",
        "faded photocopy with mild paper texture",
        "creased printout with a clipped corner",
        "low-contrast scan with light shadow near the binding edge",
    ],
)

add_category(
    "annotation_layer",
    [
        "no manual annotations",
        "yellow highlights over two key numbers",
        "red pen circle around one chart outlier",
        "blue sticky note partially covering the lower right table",
        "handwritten margin note asking for follow-up",
        "rubber stamp reading DRAFT across the header",
    ],
)

add_category(
    "numeric_context",
    [
        "include values in thousands with one decimal place",
        "include percentages, basis-point deltas, and small footnotes",
        "include dates across the next six months",
        "include currency values, totals, and year-over-year deltas",
        "include counts by region plus a total row",
    ],
)

config_builder.add_column(
    dd.ImageColumnConfig(
        name="document_image",
        prompt="""
Create a realistic single-page business document image with rich visual information.

Document requirements:
- Document type: {{ document_type }}
- Organization: {{ organization_name }}
- Document owner: {{ stakeholder.first_name }} {{ stakeholder.last_name }}, {{ stakeholder_role }}
- Owner contact line: {{ stakeholder.email_address }}; {{ stakeholder.city }}
- Intended audience: {{ audience }}
- Theme: {{ content_theme }}
- Layout style: {{ layout_style }}
- Physical/rendering condition: {{ document_condition }}
- Annotation layer: {{ annotation_layer }}
- Numeric style: {{ numeric_context }}

Required visual content:
- Primary visual: {{ primary_visual }}
- Secondary visual: {{ secondary_visual }}
- At least one readable table with row and column labels
- At least one chart, timeline, heatmap, diagram, or KPI-card cluster
- A clear title, date, organization name, document owner, section headings, and small source note
- Enough readable text to ask visual QA questions about exact values, trends, labels, owners, dates, and relationships

Make the page visually dense but professionally designed. Use realistic fonts,
alignment, legends, axis labels, table borders, captions, and spacing. The text
and numbers should be legible. Avoid blank areas, generic placeholder blocks,
or lorem ipsum. Do not include real company logos or real personal data.
""",
        model_alias=MODEL_ALIAS,
    )
)

data_designer.validate(config_builder)
print("Rich document generation config validated.")

### Preview synthetic documents

In [ ]:
preview = data_designer.preview(config_builder, num_records=5)

In [ ]:
for _ in range(len(preview.dataset)):
    preview.display_sample_record()

### Examine the dataset structure

In [ ]:
preview.dataset.head()

## Create full synthetic document dataset

In [ ]:
results = data_designer.create(
    config_builder,
    num_records=25,  # Generate 25 diverse document images
    dataset_name="synthetic-rich-documents",
)

## Export rich document seed parquet

This optional export writes `data/rich_document_seed.parquet` with a
`png_base64` column plus four orientation fields used by Notebook 2.
Use it as a seed dataset for experiments that need broad visual-document
coverage across charts, tables, annotations, and dense page layouts.


In [ ]:
import base64

import pandas as pd

DATA_DIR.mkdir(parents=True, exist_ok=True)

dataset = results.load_dataset()
base_path = results.artifact_storage.base_dataset_path

metadata_columns = [
    "primary_visual",
    "secondary_visual",
    "layout_style",
    "document_condition",
]


rows = []
for _, row in dataset.iterrows():
    img_ref = row["document_image"]
    if hasattr(img_ref, "__iter__") and not isinstance(img_ref, str):
        img_ref = img_ref[0]

    full_path = base_path / str(img_ref)
    img_bytes = full_path.read_bytes()
    output_row = {
        "png_base64": base64.b64encode(img_bytes).decode(),
    }
    output_row.update({col: row[col] for col in metadata_columns})
    rows.append(output_row)

seed = pd.DataFrame(rows)
out_path = DATA_DIR / "rich_document_seed.parquet"
seed.to_parquet(out_path, index=False)
print(f"Saved {len(seed)} rows to {out_path} ({out_path.stat().st_size / 1024:.0f} KB)")